# MFFT Ablation Study (Kaggle)
Validates Tables 4 & 5: frequency components, fusion strategies, band count, and attention mechanisms.

Tests these ablation configurations:
- **spatial_only**: Raw image feed (no DCT)
- **skip_bands=[mid,high]**: Low-frequency only
- **skip_bands=[low,high]**: Mid-frequency only
- **skip_bands=[low,mid]**: High-frequency only
- **fusion_mode=concat**: Concatenation fusion
- **fusion_mode=avg**: Average pooling fusion
- **fusion_mode=max**: Max pooling fusion
- **use_fga=False**: Without FrequencyGuidedAttention

**Prerequisite:** `train_mfft_base.ipynb` must have completed.

In [ ]:
# Cell 1: Clone repo & install deps
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/mfft_repo")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/MIHMahmudEli/ai-image-detection-research.git", str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
subprocess.run(["pip", "install", "-q", "huggingface_hub", "open_clip_torch", "scipy", "python-dotenv"], check=False)
print("Ready.")

In [ ]:
# Cell 2: Setup
import os, json, time, copy
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

In [ ]:
# Cell 3: Load manifest & create DataLoaders
from split_manifest_manager import SplitManifestManager
from kaggle_dataset_loader import KaggleDatasetLoader

manifest_mgr = SplitManifestManager(hf_token=hf_token, run_id="ablation")
manifest, manifest_sha256 = manifest_mgr.download()

loader = KaggleDatasetLoader(
    manifest=manifest, manifest_sha256=manifest_sha256,
    input_root="/kaggle/input", image_size=224,
)
train_loader, val_loader, test_loader = loader.create_dataloaders(batch_size=64, num_workers=4)
print(f"Train: {len(loader.train_data)} | Val: {len(loader.val_data)} | Test: {len(loader.test_data)}")

In [ ]:
# Cell 4: Define ablation configs
sys.path.insert(0, str(REPO_DIR / "model"))
from src.model import build_mfft

ABLATION_CONFIGS = {
    "baseline": {},
    "spatial_only": {"spatial_only": True},
    "low_freq_only": {"skip_bands": ["mid", "high"]},
    "mid_freq_only": {"skip_bands": ["low", "high"]},
    "high_freq_only": {"skip_bands": ["low", "mid"]},
    "fusion_concat": {"fusion_mode": "concat"},
    "fusion_avg": {"fusion_mode": "avg"},
    "fusion_max": {"fusion_mode": "max"},
    "no_fga": {"use_fga": False},
}
print(f"Ablation configs: {list(ABLATION_CONFIGS.keys())}")

In [ ]:
# Cell 5: Ablation training function
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

ABLATION_EPOCHS = 15

def train_ablation(name, ablation_cfg, train_loader, val_loader, test_loader, device):
    print(f"\n{'='*60}")
    print(f"Ablation: {name} | Config: {ablation_cfg}")
    print(f"{'='*60}")

    model = build_mfft("base", ablation=ablation_cfg if ablation_cfg else None).to(device)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Params: {n_params:,}")

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
    scheduler = CosineAnnealingLR(optimizer, T_max=ABLATION_EPOCHS)
    scaler = GradScaler(enabled=torch.cuda.is_available())

    best_f1 = 0
    best_state = None

    for epoch in range(1, ABLATION_EPOCHS + 1):
        model.train()
        t_loss = 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                loss = criterion(model(imgs), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            t_loss += loss.item()
        scheduler.step()

        model.eval()
        preds, labels_all, probs_all = [], [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device)
                logits = model(imgs)
                preds.extend(logits.argmax(1).cpu().numpy())
                labels_all.extend(labels.numpy())
                probs_all.extend(F.softmax(logits, dim=1).cpu().numpy())

        f1 = f1_score(labels_all, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"  Epoch {epoch:2d}/{ABLATION_EPOCHS} loss={t_loss/len(train_loader):.4f} val_f1={f1:.4f}")

    # Test
    if best_state:
        model.load_state_dict(best_state)
    model.eval()
    preds, labels_all = [], []
    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs = imgs.to(device)
            preds.extend(model(imgs).argmax(1).cpu().numpy())
            labels_all.extend(labels.numpy())
    test_f1 = f1_score(labels_all, preds, average="macro", zero_division=0)
    test_acc = accuracy_score(labels_all, preds)
    print(f"  TEST: acc={test_acc:.4f} f1={test_f1:.4f}")

    return {"name": name, "config": ablation_cfg, "n_params": n_params, "val_best_f1": best_f1, "test_acc": test_acc, "test_f1": test_f1}

In [ ]:
# Cell 6: Run all ablations
ablation_results = []
for name, cfg in ABLATION_CONFIGS.items():
    try:
        r = train_ablation(name, cfg, train_loader, val_loader, test_loader, device)
        ablation_results.append(r)
    except Exception as e:
        print(f"  FAILED {name}: {e}")

# Summary
import pandas as pd
df = pd.DataFrame(ablation_results)
print("\n=== Ablation Results ===")
print(df.sort_values("test_f1", ascending=False).to_string(index=False))

# Save & upload
out_dir = Path("/kaggle/working/ablation_results")
out_dir.mkdir(exist_ok=True)
with open(out_dir / "ablation_results.json", "w") as f:
    json.dump(ablation_results, f, indent=2, default=str)
from huggingface_hub import HfApi
api = HfApi(token=hf_token)
api.upload_file(
    path_or_fileobj=str(out_dir / "ablation_results.json"),
    path_in_repo="results/ablation_results.json",
    repo_id="MohsinElis/mfft-checkpoints", repo_type="model",
)
print("Uploaded to HF.")